# Upstage AIRE LLM Deep Learning Coding Test

## Overview

This notebook is designed to assess whether the candidate can improve a specific benchmark score by fine-tuning a base LLM and whether they have the relevant engineering skills to do so.

## Task Description
* In this deep learning coding test, you are required to improve an LLM benchmark score by using Mistral-7B as a base model. You can utilize a combination of various fine-tuning strategies, including a combination of multiple datasets, Parameter Efficient Fine-Tuning (PEFT), and finding the optimal configuration of hyperparameters. The following are examples of available improvement strategies:
  * You can change the instruction template and `Tokenizer` config, etc.
  * You can implement custom classes, e.g., `CustomDataset`, `CustomCollator` and `CustomTrainer` tailored for improved instruction tuning.
  * You can explore and select high-quality datasets, or you can synthesize datasets by leveraging large-scale models (>7B).
  * You can find the optimal configuration of hyperparameters.

* We recommend using a PEFT method like QLoRA to avoid OOM errors. 7B LLMs with QLoRA-based PEFT can be trained by using a single T4 GPU (e.g., Google Colab). Note that we generally expect to achieve higher scores when using full fine-tuning compared to using PEFT.
* The notebook and source code you submit must be reproducible. For reproducibility and correct assessment, please describe the methodology you used to improve the benchmark score in as much detail as possible.
* Regarding the coding test assessment, while the primary goal is to boost your benchmark score, your methodology's logical consistency and academic rationale will also be evaluated. For example, you may fail to improve your benchmark score. Still, you may receive high marks if you analyze the reasons for your failure from both a technical and academic perspective and produce meaningful results. For more information on the benchmark score, see the Evaluation Protocol below.
* (__Important!__) The code below is provided as an example and does not guarantee the correctness of implementation and execution. You can build upon it or use a completely different code base.




## Evaluation Protocol
* ARC (AI2 Reasoning Challenge): The AI2’s Reasoning Challenge (ARC) dataset is a multiple-choice question-answering dataset, containing questions from science exams from grade 3 to grade 9. The dataset is split in two partitions: Easy and Challenge, where the latter partition contains the more difficult questions that require reasoning. Most of the questions have 4 answer choices, with $<$1\% of all the questions having either 3 or 5 answer choices. ARC includes a supporting KB of 14.3M unstructured text passages. [[source]](https://arxiv.org/abs/1911.07176)
  * Clark, Peter, et al. "Think you have solved question answering? try arc, the ai2 reasoning challenge." arXiv preprint arXiv:1803.05457 (2018). [[pdf]](https://arxiv.org/pdf/1803.05457v1.pdf)
  * The ARC Challenge benchmark has been used as an essential benchmark for major LLMs as they are released, including OpenAI's GPT-4, Google's Gemini, and Meta's LLaMA.

    source: Jiang, Albert Q., et al. "Mixtral of experts." arXiv preprint arXiv:2401.04088 (2024). [[pdf]](https://arxiv.org/pdf/2401.04088.pdf)

  * In our preliminary experiment, Mistral-7B, a base model in this notebook, has shown the ARC challenge score of 61.43.
  * In this notebook, we will utilize EleutherAI's [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) library to evaluate the ARC challenge score.


## Submission
* Submit a notebook file and your final model (including ckpt and tokenizer). We recommend submitting the notebook as an ipynb file or Google colab, and the final model should be uploaded and shared on the HuggingFace Hub.
* Provide a separate report/documentation in PDF format where you performed detailed analysis, etc.

## Enviroment Setting

In [ ]:
# Required when training models/data that are gated on Hugging Face, and required for pushing models to Hugging Face.
!pip install huggingface_hub -q -U
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# (Optional) Mount Google Drive. If you are not using Colab, please comment out the code below.
from google.colab import drive
drive.mount('/gdrive', force_remount=True)
drive.mount('/content/drive')

Mounted at /gdrive
Mounted at /content/drive


In [ ]:
# (Optional) 구글 드라이브를 사용할 경우 아래의 코드를 통해 모델을 캐싱하여 시간을 절약하고 학습 데이터를 드라이브에 저장할 수 있습니다.
# If you're running Jupyter notebook in local, set your local caching directory in `cache_dir`.

# https://stackoverflow.com/questions/56081324/why-are-google-colab-shell-commands-not-working
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

import os
cache_dir = "/content/drive/MyDrive/huggingface_cache"
os.makedirs(cache_dir, exist_ok=True)  # Ensure the directory exists.

In [ ]:
# 구글 드라이브를 사용하지 않는 경우, 아래의 코드에서 오류가 발생하는 것을 방지하기 위해 아래의 코드를 실행하세요.
# If you are not using Google Drive, please run the code below to prevent errors in the code below.
import os

cache_dir = "/tmp/huggingface_cache"

## Package & Library Install

In [ ]:
!python -m pip install --upgrade pip -q -U
!pip install -q datasets
!pip install -q -U scipy
!pip install -q -U transformers
!pip install -q -U Jinja2
!pip install -q -U trl
!pip install -q -U bitsandbytes
!pip install -q -U peft
!pip install -q -U accelerate
!pip install flash-attn -q -U

In [ ]:
model_id = "mistralai/Mistral-7B-v0.1"

local_path = model_id
local_save_path = os.path.join(cache_dir, local_path)

## Download Base Model



In [ ]:
from huggingface_hub import snapshot_download
import os

def download_model_repo(repo_id, local_dir):
    # Download the whole repository to the specified local directory.
    repo_path = snapshot_download(repo_id=repo_id,
                                  cache_dir=local_dir,
                                  local_dir=local_dir,
                                  local_dir_use_symlinks=False)

    print(f"Repository is saved to: {repo_path}")

def main():
    download_model_repo(model_id, local_save_path)
    print()

if __name__ == "__main__":
    main()

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Repository is saved to: /content/drive/MyDrive/huggingface_cache/mistralai/Mistral-7B-v0.1



## Load Dataset

In [ ]:
from datasets import load_dataset

alpaca_train_ds = load_dataset("tatsu-lab/alpaca", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# Print the first row of the dataset.
alpaca_train_ds[0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}

In [ ]:
# Randomly sample 500 examples.
sampled_alpaca_train_ds = alpaca_train_ds.shuffle(seed=42).select(range(500))

In [ ]:
sampled_alpaca_train_ds

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 500
})

In [ ]:
import json

# This function is used to output the appropriate instruction format for each row in the dataset.
# You can use your own instruction template (format).
def convert_text_to_formatted_text(example_batch):
    converted_text = f"""<s>[INST] {example_batch["instruction"]} here are the inputs {example_batch["input"]} [/INST] \\n {example_batch["output"]} </s>"""
    return {
        "instruction": example_batch["instruction"],
        "input": example_batch["input"],
        "output": example_batch["output"],
        "text": converted_text
    }

# Apply the function to the dataset.
sampled_alpaca_train_ds = sampled_alpaca_train_ds.map(convert_text_to_formatted_text)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
sampled_alpaca_train_ds[0]

{'instruction': 'What would be the best type of exercise for a person who has arthritis?',
 'input': '',
 'output': 'For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.',
 'text': '<s>[INST] What would be the best type of exercise for a person who has arthritis? here are the inputs  [/INST] \\n For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis. </s>'}

## Load Base Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Activate 4-bit precision base model loading.
use_4bit = True

# Activate nested quantization for 4-bit base models.
use_nested_quant = False

# Pick exactly one mixed-precision mode end-to-end.
# bf16 does not use GradScaler; fp16 does.
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16
bnb_4bit_compute_dtype = "bfloat16" if bf16 else "float16"

# Quantization type (fp4 or nf4).
bnb_4bit_quant_type = "nf4"

# Load tokenizer and model with QLoRA configuration.
compute_dtype = torch.bfloat16 if bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16, you can accelerate training with the argument --bf16")
        print("=" * 80)


model = AutoModelForCausalLM.from_pretrained(
    local_save_path,
    quantization_config=bnb_config,  # Comment out to do a full fine-tune.
    device_map='auto',
    # attn_implementation="flash_attention_2",  # Comment out if you are not using an Ampere GPU (e.g. A100, H100, A6000).
    torch_dtype=compute_dtype,  # Set to torch.float16 if you are not using an Ampere GPU (e.g. A100, H100, A6000).
    cache_dir=cache_dir)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_id, add_eos_token=True)
tokenizer.padding_side = "right"

Your GPU supports bfloat16, you can accelerate training with the argument --bf16


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# If the tokenizer has <unk>, set the PAD token to <unk> or set it to an EOS token.
if '<pad>' in tokenizer.get_vocab():
    print('<pad> token is in the tokenizer. Using <pad> for pad')
    # Set the pad token.
    tokenizer.pad_token = '<pad>'
elif '<unk>' in tokenizer.get_vocab():
    print('<unk> token is in the tokenizer. Using unk for pad')
    # Set the pad token.
    tokenizer.pad_token = '<unk>'
else:
    print(f'Using EOS token, {tokenizer.eos_token}, for padding')
    tokenizer.pad_token = tokenizer.eos_token

<unk> token is in the tokenizer. Using unk for pad


In [ ]:
model.pad_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

assert model.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"

print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)
print('Number of tokens now in tokenizer:', tokenizer.vocab_size)

Tokenizer pad token ID: 0
Model pad token ID: 0
Model config pad token ID: 0
Number of tokens now in tokenizer: 32000


## Set Up LoRA

In [ ]:
def print_trainable_parameters(model):
    """
    Outputs the number of trainable parameters in the model and the total number of parameters.
    This allows you to see the size of your model and the percentage of parameters used for training.
    """
    trainable_params = 0
    total_params = 0

    for _, param in model.named_parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    trainable_percent = 100 * trainable_params / total_params

    print(f"Trainable Parameters: {trainable_params}")
    print(f"Total Parameters: {total_params}")
    print(f"Trainable %: {trainable_percent:.2f}")

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model

from peft import prepare_model_for_kbit_training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    # target_modules = ["gate_proj", "down_proj", "up_proj", "q_proj", "v_proj", "k_proj", "o_proj"]  # Targeting all modules for more intensive training.
    target_modules = ["q_proj", "v_proj"],  # There are options to deepen the fine-tuning by unfreezing more weights but with a cost in performance.
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

ft_model = get_peft_model(
    model,
    config,
)
print_trainable_parameters(ft_model)

Trainable Parameters: 3407872
Total Parameters: 3755479040
Trainable %: 0.09


## Set Hyperparameters

In [ ]:
from trl import SFTConfig, SFTTrainer

# pad_token이 없을 수 있음
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

training_arguments = SFTConfig(
    output_dir=os.path.join(cache_dir, "results"),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=bf16,
    fp16=fp16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)


import trl, transformers, peft, accelerate, datasets, torch
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
print("datasets", datasets.__version__)
print("torch", torch.__version__)


In [ ]:
# Setting sft parameters.
trainer = SFTTrainer(
    model=ft_model,
    train_dataset=sampled_alpaca_train_ds,
    processing_class=tokenizer,
    args=training_arguments,
)


/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:225: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Fine-Tuning

In [ ]:
import torch
from collections import Counter

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

print("args fp16:", trainer.args.fp16)
print("args bf16:", trainer.args.bf16)

try:
    print("accelerate mixed_precision:", trainer.accelerator.state.mixed_precision)
    print("scaler:", trainer.accelerator.scaler)
except Exception as e:
    print(e)

cnt = Counter()
for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        cnt[str(p.dtype)] += p.numel()
        print(name, p.dtype, p.shape)
print(cnt)


/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
10,1.801500
20,1.676200
30,1.563900
40,1.349800
50,1.437600
60,1.431600
70,1.286500
80,1.333100
90,1.398300
100,1.348600


Checkpoint destination directory /content/drive/MyDrive/huggingface_cache/results/checkpoint-50 already exists and is non-empty. Saving will proceed but saved results may be invalid.
/usr/local/lib/python3.10/dist-packages/peft/utils/save_and_load.py:154: UserWarning: Could not find a config file in /content/drive/MyDrive/huggingface_cache/mistralai/Mistral-7B-v0.1 - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
Checkpoint destination directory /content/drive/MyDrive/huggingface_cache/results/checkpoint-100 already exi

TrainOutput(global_step=125, training_loss=1.4462912139892579, metrics={'train_runtime': 87.0375, 'train_samples_per_second': 5.745, 'train_steps_per_second': 1.436, 'total_flos': 2252541568843776.0, 'train_loss': 1.4462912139892579, 'epoch': 1.0})

trainer.train()

## Save and Push Model to HuggingFace Hub

In [ ]:
merged_model = trainer.model.merge_and_unload()

/usr/local/lib/python3.10/dist-packages/peft/tuners/lora/bnb.py:272: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
new_model_name = "{YOUR_HF_ID}/mistral-7b-qlora-alpaca-sample-0.5k"  # Please specify your own repo/model id.

In [ ]:
output_merged_dir = os.path.join(cache_dir, new_model_name)
merged_model.save_pretrained(output_merged_dir, safe_serialization=True)
tokenizer.save_pretrained(output_merged_dir)

('/content/drive/MyDrive/huggingface_cache/YOUR_HF_ID/mistral-7b-qlora-alpaca-sample-0.5k/tokenizer_config.json',
 '/content/drive/MyDrive/huggingface_cache/YOUR_HF_ID/mistral-7b-qlora-alpaca-sample-0.5k/special_tokens_map.json',
 '/content/drive/MyDrive/huggingface_cache/YOUR_HF_ID/mistral-7b-qlora-alpaca-sample-0.5k/tokenizer.model',
 '/content/drive/MyDrive/huggingface_cache/YOUR_HF_ID/mistral-7b-qlora-alpaca-sample-0.5k/added_tokens.json',
 '/content/drive/MyDrive/huggingface_cache/YOUR_HF_ID/mistral-7b-qlora-alpaca-sample-0.5k/tokenizer.json')

In [ ]:
# Push merged model to the HF hub.
merged_model.push_to_hub(repo_id=new_model_name, token=True, max_shard_size="5GB", safe_serialization=True)
tokenizer.push_to_hub(repo_id=new_model_name, token=True)

model.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/YOUR_HF_ID/mistral-7b-qlora-alpaca-sample-0.5k/commit/f388f41a947fd4377218f5f56215ed6435de8e40', commit_message='Upload tokenizer', commit_description='', oid='f388f41a947fd4377218f5f56215ed6435de8e40', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
del model, merged_model
torch.cuda.empty_cache()

## ARC Challenge Evaluation

If you get an OOM error in the evaluation command below when using Google Colab, try restarting your session and rerunning the cells, except for training parts.

In [ ]:
!git clone https://github.com/EleutherAI/lm-evaluation-harness
!cd lm-evaluation-harness && pip install -e .

fatal: destination path 'lm-evaluation-harness' already exists and is not an empty directory.
Obtaining file:///content/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lm_eval (pyproject.toml) ... done
  Created wheel for lm_eval: filename=lm_eval-0.4.1-0.editable-py3-none-any.whl size=14928 sha256=61b7827ca3df9ae3dc1fa7b69c29bd4b4132d321f0be660a1ea980b310922bd2
  Stored in directory: /tmp/pip-ephem-wheel-cache-gahd_1wg/wheels/dc/8d/a0/ce1a137b6a29fcf5007da91566ee423695e01d20703991091d
Successfully built lm_eval
  Attempting uninstall: lm_eval
    Found existing installation: lm_eval 0.4.1
    Uninstalling lm_eval-0.4.1:
      Successfully uninstalled lm_eval-0.4.1


In [ ]:
# Mistral-7B's ARC Challenge Score: 61.43
# eval_output_path = os.path.join(cache_dir, "results", "arc_challenge")
# os.makedirs(eval_output_path, exist_ok=True)

# # It takes about 11 minutes on a single A100 40GB GPU (about 100 minutes on a single T4 GPU)
# eval_output_path = os.path.join(eval_output_path, "result.json")
# tasks = "arc_challenge"

# eval_cmd = f"""
# lm_eval --model hf \
#     --model_args pretrained=mistralai/Mistral-7B-v0.1,trust_remote_code=True,dtype=float16 \
#     --tasks {tasks} \
#     --device cuda:0 \
#     --batch_size 8 \
#     --num_fewshot 25 \
#     --output_path {eval_output_path}
# """

"""
hf (pretrained=mistralai/Mistral-7B-v0.1,trust_remote_code=True,dtype=float16), gen_kwargs: (None), limit: None, num_fewshot: 25, batch_size: 8
|    Tasks    |Version|Filter|n-shot| Metric |Value |   |Stderr|
|-------------|------:|------|-----:|--------|-----:|---|-----:|
|arc_challenge|      1|none  |    25|acc     |0.5700|±  |0.0145|
|             |       |none  |    25|acc_norm|0.6143|±  |0.0142|
"""

In [ ]:
eval_output_path = os.path.join(cache_dir, "results", "arc_challenge")
os.makedirs(eval_output_path, exist_ok=True)

# It takes about 11 minutes on a single A100 40GB GPU (about 100 minutes on a single T4 GPU).
eval_output_path = os.path.join(eval_output_path, "result-25shot.json")
tasks = "arc_challenge"

eval_cmd = f"""
lm_eval --model hf \
    --model_args pretrained={new_model_name},trust_remote_code=True,dtype=float16 \
    --tasks {tasks} \
    --device cuda:0 \
    --batch_size 8 \
    --num_fewshot 25 \
    --output_path {eval_output_path}
"""

In [ ]:
# Run an evaluation command.
!{eval_cmd}

2024-03-02 14:12:06.938304: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-02 14:12:06.938356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-02 14:12:06.940334: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-02 14:12:08.132887: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-03-02:14:12:12,215 INFO     [__main__.py:217] Verbosity set to INFO
2024-03-02:14:12:12,215 INFO     [__init__.py:369] lm_eval.tasks.initialize_tasks() is deprecated and no longer necessary. It will be removed in v0.4.2 release. TaskMan

The final model shows the ARC Challenge score of 58.19 (the value corresponding to acc_norm). This is worse than the pre-fine-tuning score of 61.43 (in our preliminary study), indicating that a better strategy is needed to improve the score.

In [ ]:
!pip freeze > requirements.txt

In [ ]:
!cat requirements.txt

absl-py==1.4.0
accelerate==0.27.2
aiohttp==3.9.3
aiosignal==1.3.1
alabaster==0.7.16
albumentations==1.3.1
altair==4.2.2
annotated-types==0.6.0
anyio==3.7.1
appdirs==1.4.4
argon2-cffi==23.1.0
argon2-cffi-bindings==21.2.0
array-record==0.5.0
arviz==0.15.1
astropy==5.3.4
astunparse==1.6.3
async-timeout==4.0.3
atpublic==4.0
attrs==23.2.0
audioread==3.0.1
autograd==1.6.2
Babel==2.14.0
backcall==0.2.0
beautifulsoup4==4.12.3
bidict==0.23.1
bigframes==0.21.0
bitsandbytes==0.42.0
bleach==6.1.0
blinker==1.4
blis==0.7.11
blosc2==2.0.0
bokeh==3.3.4
bqplot==0.12.43
branca==0.7.1
build==1.0.3
CacheControl==0.14.0
cachetools==5.3.3
catalogue==2.0.10
certifi==2024.2.2
cffi==1.16.0
chardet==5.2.0
charset-normalizer==3.3.2
chex==0.1.85
click==8.1.7
click-plugins==1.1.1
cligj==0.7.2
cloudpathlib==0.16.0
cloudpickle==2.2.1
cmake==3.27.9
cmdstanpy==1.2.1
colorama==0.4.6
colorcet==3.0.1
colorlover==0.3.0
colour==0.1.5
community==1.0.0b1
confection==0.1.4
cons==0.4.6
contextlib2==21.6.0
contourpy==1.2.0
cryp

## Content License

Copyright : <font color='blue'> <b> ©2024 by Upstage Co., Ltd. All rights reserved.</font></b>

<font color='red'><b>WARNING</font> : 본 콘텐츠의 지식재산권은 업스테이지에 귀속됩니다. 본 콘텐츠를 어떠한 경로로든 외부로 유출 및 수정하는 행위를 엄격히 금합니다. </b>